# Continuous Control

---

In this notebook, you will learn how to use the Unity ML-Agents environment for the second project of the [Deep Reinforcement Learning Nanodegree](https://www.udacity.com/course/deep-reinforcement-learning-nanodegree--nd893) program.

### 1. Start the Environment

We begin by importing the necessary packages.  If the code cell below returns an error, please revisit the project instructions to double-check that you have installed [Unity ML-Agents](https://github.com/Unity-Technologies/ml-agents/blob/master/docs/Installation.md) and [NumPy](http://www.numpy.org/).

In [ ]:
from unityagents import UnityEnvironment
import numpy as np
import torch
import matplotlib.pyplot as plt
from collections import deque
%matplotlib inline

Next, we will start the environment! We have two versions available:
- **Version 1**: Single agent — `Reacher_Version_1/Reacher.exe`
- **Version 2**: 20 agents — `Reacher_Version_2/Reacher.exe`

Select the version you want to use below.

In [ ]:
# Choose version: 1 (single agent) or 2 (20 agents)
VERSION = 1

if VERSION == 1:
    env = UnityEnvironment(file_name='Reacher_Version_1/Reacher.exe')
else:
    env = UnityEnvironment(file_name='Reacher_Version_2/Reacher.exe')

Environments contain **_brains_** which are responsible for deciding the actions of their associated agents. Here we check for the first brain available, and set it as the default brain we will be controlling from Python.

In [ ]:
# get the default brain
brain_name = env.brain_names[0]
brain = env.brains[brain_name]

### 2. Examine the State and Action Spaces

In this environment, a double-jointed arm can move to target locations. A reward of `+0.1` is provided for each step that the agent's hand is in the goal location. Thus, the goal of your agent is to maintain its position at the target location for as many time steps as possible.

The observation space consists of `33` variables corresponding to position, rotation, velocity, and angular velocities of the arm.  Each action is a vector with four numbers, corresponding to torque applicable to two joints.  Every entry in the action vector must be a number between `-1` and `1`.

Run the code cell below to print some information about the environment.

In [ ]:
# reset the environment
env_info = env.reset(train_mode=True)[brain_name]

# number of agents
num_agents = len(env_info.agents)
print('Number of agents:', num_agents)

# size of each action
action_size = brain.vector_action_space_size
print('Size of each action:', action_size)

# examine the state space 
states = env_info.vector_observations
state_size = states.shape[1]
print('There are {} agents. Each observes a state with length: {}'.format(states.shape[0], state_size))
print('The state for the first agent looks like:', states[0])

### 3. Take Random Actions in the Environment

In the next code cell, you will learn how to use the Python API to control the agent and receive feedback from the environment.

Once this cell is executed, you will watch the agent's performance, if it selects an action at random with each time step.  A window should pop up that allows you to observe the agent, as it moves through the environment.  

Of course, as part of the project, you'll have to change the code so that the agent is able to use its experience to gradually choose better actions when interacting with the environment!

In [ ]:
env_info = env.reset(train_mode=False)[brain_name]     # reset the environment    
states = env_info.vector_observations                  # get the current state (for each agent)
scores = np.zeros(num_agents)                          # initialize the score (for each agent)
while True:
    actions = np.random.randn(num_agents, action_size) # select an action (for each agent)
    actions = np.clip(actions, -1, 1)                  # all actions between -1 and 1
    env_info = env.step(actions)[brain_name]           # send all actions to tne environment
    next_states = env_info.vector_observations         # get next state (for each agent)
    rewards = env_info.rewards                         # get reward (for each agent)
    dones = env_info.local_done                        # see if episode finished
    scores += env_info.rewards                         # update the score (for each agent)
    states = next_states                               # roll over states to next time step
    if np.any(dones):                                  # exit loop if episode finished
        break
print('Total score (averaged over agents) this episode: {}'.format(np.mean(scores)))

### 4. Train the DDPG Agent

We use a DDPG (Deep Deterministic Policy Gradient) agent with batch normalization, gradient clipping, and delayed learning updates. The agent works for both Version 1 (single agent) and Version 2 (20 agents).

Key hyperparameters:
- Buffer size: 1,000,000 | Batch size: 128
- Actor LR: 1e-4 | Critic LR: 1e-3
- Learn every 20 steps, 10 updates per learning step
- Ornstein-Uhlenbeck noise (θ=0.15, σ=0.2)

In [ ]:
from ddpg_agent import Agent

# Create the DDPG agent
agent = Agent(state_size=state_size, action_size=action_size, random_seed=42)

print(f'Device: {torch.device("cuda:0" if torch.cuda.is_available() else "cpu")}')
print(f'Training Version {VERSION} with {num_agents} agent(s)')

In [ ]:
def train_ddpg(env, agent, brain_name, num_agents, n_episodes=300, max_t=1000):
    """Train DDPG agent. Works for both single and multi-agent environments."""
    scores_all = []
    scores_window = deque(maxlen=100)
    solved = False

    for i_episode in range(1, n_episodes + 1):
        env_info = env.reset(train_mode=True)[brain_name]
        states = env_info.vector_observations
        agent.reset()
        scores = np.zeros(num_agents)

        for t in range(max_t):
            actions = agent.act(states)
            env_info = env.step(actions)[brain_name]
            next_states = env_info.vector_observations
            rewards = env_info.rewards
            dones = env_info.local_done

            agent.step(states, actions, rewards, next_states, dones)
            states = next_states
            scores += np.array(rewards)

            if np.any(dones):
                break

        episode_score = np.mean(scores)
        scores_all.append(episode_score)
        scores_window.append(episode_score)
        avg_score = np.mean(scores_window)

        if i_episode % 10 == 0:
            print(f'\rEpisode {i_episode}\tAverage Score: {avg_score:.2f}\tLast Score: {episode_score:.2f}')

        if avg_score >= 30.0 and len(scores_window) >= 100 and not solved:
            solved = True
            print(f'\nEnvironment solved in {i_episode} episodes!\tAverage Score: {avg_score:.2f}')
            torch.save(agent.actor_local.state_dict(), f'checkpoint_actor_v{VERSION}_solved.pth')
            torch.save(agent.critic_local.state_dict(), f'checkpoint_critic_v{VERSION}_solved.pth')

    return scores_all

# Train the agent
scores = train_ddpg(env, agent, brain_name, num_agents, n_episodes=300, max_t=1000)

In [ ]:
# Save final checkpoints
torch.save(agent.actor_local.state_dict(), 'checkpoint_actor.pth')
torch.save(agent.critic_local.state_dict(), 'checkpoint_critic.pth')
torch.save(agent.actor_local.state_dict(), f'checkpoint_actor_v{VERSION}.pth')
torch.save(agent.critic_local.state_dict(), f'checkpoint_critic_v{VERSION}.pth')
print('Checkpoints saved.')

In [ ]:
# Plot the scores
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(np.arange(1, len(scores) + 1), scores, label='Score per Episode', alpha=0.6)

if len(scores) >= 100:
    rolling_avg = [np.mean(scores[max(0, i - 100):i + 1]) for i in range(len(scores))]
    ax.plot(np.arange(1, len(scores) + 1), rolling_avg, label='100-Episode Average', linewidth=2)

ax.axhline(y=30.0, color='r', linestyle='--', label='Solved Threshold (+30)')
ax.set_xlabel('Episode #')
ax.set_ylabel('Score')
ax.set_title(f'DDPG Training Scores - Reacher Version {VERSION}')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'scores_plot_v{VERSION}.png', dpi=150)
plt.show()
print(f'Plot saved to scores_plot_v{VERSION}.png')

### 5. Watch a Trained Agent

Load the saved checkpoint and watch the trained agent perform in the environment.

In [ ]:
# Load trained weights (uncomment and adjust path if loading from a previous run)
# agent.actor_local.load_state_dict(torch.load(f'checkpoint_actor_v{VERSION}.pth'))

env_info = env.reset(train_mode=False)[brain_name]
states = env_info.vector_observations
scores = np.zeros(num_agents)

for t in range(1000):
    actions = agent.act(states, add_noise=False)
    env_info = env.step(actions)[brain_name]
    next_states = env_info.vector_observations
    rewards = env_info.rewards
    dones = env_info.local_done
    scores += np.array(rewards)
    states = next_states
    if np.any(dones):
        break

print(f'Score (averaged over agents): {np.mean(scores):.2f}')

In [ ]:
env.close()